In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, ArrayType
from pyspark.sql.functions import explode, lower, col, collect_set

# In cluster use the HDFS path prefix
path_prefix = "hdfs:///projects/BDA-12/"
# In local use the local path prefix
# path_prefix = "" 

spark = SparkSession.builder \
    .appName("IngredientFrequency") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Strict schema
ingredients_schema = StructType([
    StructField('fdc_id', LongType(), False),
    StructField('description', StringType(), True),
    StructField('all_ingredients', ArrayType(StringType()), True)
])

df = spark.read.schema(ingredients_schema).parquet(
    f'{path_prefix}/output/ingredients_nutrional_profiles/'
)

In [ ]:
# Explode ingredients and normalize
from pyspark.sql.functions import trim, regexp_replace
pattern = r'[\(\)\"\*\[\]\{\}\.,:;\'\-&]' #pattern
exploded = df.select('fdc_id', 'description', explode('all_ingredients').alias('ingredient'))
exploded = exploded.withColumn('ingredient_norm', trim(lower(regexp_replace(col('ingredient'), pattern, ''))))
# Remove duplicates: ingredient counted once per food
unique_ingredients = exploded.groupBy('fdc_id', 'description').agg(collect_set('ingredient_norm').alias('ingredients_set'))
exploded_unique = unique_ingredients.select('fdc_id', 'description', explode('ingredients_set').alias('ingredient_norm'))

In [ ]:
from pyspark.sql.functions import count
if "hdfs" not in path_prefix:
    import os
    os.makedirs(f'{path_prefix}/output/ingredient_frequency', exist_ok=True)
ingredient_counts = exploded_unique.groupBy('ingredient_norm').agg(
    count('fdc_id').alias('food_count'),
    collect_set('description').alias('examples')
)
ingredient_counts = ingredient_counts.orderBy(col('food_count').desc())
# Top and rare as Spark DataFrames (no pandas)
top10_df = ingredient_counts.limit(10)
rare10_df = ingredient_counts.orderBy(col('food_count').asc()).limit(10)
# Display results using Spark's show()
top10_df.show(truncate=False)
rare10_df.show(truncate=False)
# Save all ingredient frequencies to CSV directory using Spark (creates a folder)
ingredient_counts.coalesce(1).write.mode("overwrite").option("header", "true").csv(f'{path_prefix}/output/ingredient_frequency')